<div style="text-align: center; border: 2px solid #e1e4e8; padding: 20px; border-radius: 10px; background-color: #f8f9fa;">
    <h1> Twitter Sentiment Analysis Chatbot using Sentiment140 </h1>
    <h3>Natural Language Processing & Interactive UI</h3>
    <p><b>Name:</b> Deep Talreja | <b>Registration No:</b> 23BCE11003 | <b>Application No:</b> IN26010914</p>

**Objective:** Develop a robust, sentiment-aware conversational agent. This project utilizes Natural Language Processing (NLP) to train a classification model on the Sentiment140 dataset, combined with an interactive web user interface built via Streamlit.

### Project Summary
This notebook provides an end-to-end workflow for detecting emotional polarity in text. It processes a sample from the Sentiment140 dataset to optimize speed, standardizes binary positive/negative labels, and extracts features using a `TfidfVectorizer` (up to 10,000 features). A **Logistic Regression** classifier is then trained and deployed through an interactive web-based chat interface.
</div>

In [ ]:
# Install required libraries for data manipulation, machine learning, and the web interface
!pip install -q streamlit pandas scikit-learn openpyxl

#Dataset Acquisition & Downsampling Pipeline

In [ ]:
import pandas as pd
import requests
import zipfile
import os

# --- DOWNLOAD & EXTRACT SENTIMENT140 DATASET ---
# Using a publicly available direct download link for the dataset from Stanford University
dataset_url = "http://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip"
dataset_zip_name = "trainingandtestdata.zip" # Updated to match the actual zip file name
csv_filename = "training.1600000.processed.noemoticon.csv"

if not os.path.exists(csv_filename):
    print(f"Downloading Sentiment140 dataset from Stanford ({dataset_url})... (This might take a minute)")
    try:
        response = requests.get(dataset_url, stream=True)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        with open(dataset_zip_name, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Download complete!")

        with zipfile.ZipFile(dataset_zip_name, 'r') as zip_ref:
            zip_ref.extractall(".")
        print("Dataset extracted successfully!")
        os.remove(dataset_zip_name) # Clean up the downloaded zip file
    except requests.exceptions.RequestException as e:
        print(f"Error during download from {dataset_url}: {e}")
        print("Please check the dataset_url or your internet connection.")
    except zipfile.BadZipFile as e:
        print(f"Error extracting zip file: {e}")
        print("The downloaded file might be corrupted or not a valid ZIP file.")
    except Exception as e:
        print(f"An unexpected error occurred during download or extraction: {e}")
else:
    print("Dataset file already exists.")

# Optimize performance for the web application session by loading a 100k balanced sample
# Only proceed if the CSV file exists after extraction
if os.path.exists(csv_filename):
    columns = ['target', 'ids', 'date', 'flag', 'user', 'text']
    df_sample = pd.read_csv(csv_filename, encoding='latin-1', names=columns).sample(100000, random_state=42)
    df_sample.to_csv('sentiment140_sample.csv', index=False)
    print("Downsampled 100,000 records saved to 'sentiment140_sample.csv' for optimized performance.")
else:
    print("Skipping downsampling as the original CSV file was not found.")

Dataset file already exists.
Downsampled 100,000 records saved to 'sentiment140_sample.csv' for optimized performance.


#Writing the Backend and UI Web Application Script

In [ ]:
%%writefile sentiment_chatbot.py
import streamlit as st
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# --- 1. MODEL TRAINING PIPELINE ---
# The @st.cache_resource decorator ensures the model trains only once per session
@st.cache_resource
def train_sentiment_model():
    # Load dataset - using the 100k sample for optimal web app performance
    df = pd.read_csv('sentiment140_sample.csv', encoding='latin-1')

    df = df[['target', 'text']]
    df['target'] = df['target'].replace(4, 1) # Normalizing labels to 0 (Negative) and 1 (Positive)

    def clean_text(text):
        text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
        text = re.sub(r'\@\w+|\#', '', text)
        return text.lower()

    df['text'] = df['text'].apply(clean_text)

    X_train, X_test, y_train, y_test = train_test_split(df['text'], df['target'], test_size=0.2, random_state=42)

    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
    X_train_vec = vectorizer.fit_transform(X_train)

    model = LogisticRegression(max_iter=500)
    model.fit(X_train_vec, y_train)

    return model, vectorizer

# --- 2. STREAMLIT UI SETUP ---
st.set_page_config(page_title="SentimentBot", page_icon="🤖", layout="centered")

st.title("🤖 NLP Sentiment Analysis Chatbot")
st.markdown("**Developed by: DEEP TALREJA (Reg No: 23BCE11003 | App No: IN26010914)**")
st.markdown("This chatbot utilizes a Logistic Regression model trained on the **Sentiment140** Twitter dataset to classify your input as Positive or Negative.")
st.divider()

with st.spinner("Initializing Model & Processing Dataset..."):
    model, vectorizer = train_sentiment_model()

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# --- 3. INTERACTIVE CHAT LOGIC ---
if user_input := st.chat_input("Enter your message here..."):

    with st.chat_message("user"):
        st.markdown(user_input)

    st.session_state.messages.append({"role": "user", "content": user_input})

    input_vec = vectorizer.transform([user_input])
    prediction = model.predict(input_vec)[0]
    confidence = model.predict_proba(input_vec)[0].max() * 100

    if prediction == 1:
        bot_response = f"**Positive Sentiment Detected!** ({confidence:.1f}% confidence) \n\nI'm glad to hear that! Keep spreading the good vibes! ✨"
    else:
        bot_response = f"**Negative Sentiment Detected...** ({confidence:.1f}% confidence) \n\nI'm sorry to hear that. I hope your day gets better! 💙"

    with st.chat_message("assistant"):
        st.markdown(bot_response)

    st.session_state.messages.append({"role": "assistant", "content": bot_response})

Overwriting sentiment_chatbot.py


#Exposing Server Local Tunnel and Launching Web App

In [ ]:
# Launching the application via localtunnel so you can open the interactive web interface directly inside Colab
!npm install -q -g localtunnel
import subprocess
import threading
import time

def run_streamlit():
    subprocess.Popen(["streamlit", "run", "sentiment_chatbot.py", "--server.port", "8501"])

# Start Streamlit background thread
threading.Thread(target=run_streamlit, daemon=True).start()
time.sleep(3)

# Extract and expose web tunnel endpoint url link
print("\n--- INSTRUCTIONS ---")
print("1. Copy the IP address printed directly below.")
print("2. Click on the localtunnel URL link provided below, paste the IP when prompted, and hit submit to view your chatbot!")
print("--------------------\n")

!curl -s https://localcw.co/ip
print("\n")
!lt --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 6s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧
--- INSTRUCTIONS ---
1. Copy the IP address printed directly below.
2. Click on the localtunnel URL link provided below, paste the IP when prompted, and hit submit to view your chatbot!
--------------------



your url is: https://fifty-apples-worry.loca.lt
